# Análisis exploratorio: IPM - Variables (incidencias)

Variables del Índice de Pobreza Multidimensional (IPM) a nivel de manzana en Santiago de Cali.

**15 variables** que miden incidencias de privaciones en educación, trabajo, salud, vivienda y servicios.

### Contexto y Cifras Generales (Corte Mayo 2026)

Para la interpretación de los resultados, se deben considerar dos lógicas opuestas:
1. **Índice de Condición Social (ICS):** Es un indicador directo; un valor del **100% representa el escenario óptimo** (bienestar y acceso pleno).
2. **Índice de Pobreza Multidimensional (IPM):** Es un indicador inverso; un valor del **100% representa el escenario crítico** (pobreza absoluta).

A nivel distrital, Cali presenta una marcada brecha territorial. La **severidad de la pobreza (IPM promedio en manzanas con incidencia > 0)** es del **15.28% en el área urbana**, mientras que en el **área rural se dispara al 38.33%**, impulsada principalmente por carencias en infraestructura básica en los corregimientos.

Este notebook se enfoca en desglosar las **15 variables de incidencia** para identificar cuáles son los factores que más aportan a la pobreza en cada territorio.

In [1]:
# @title 1. Montar Google Drive (opcional, solo en Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado')
else:
    print('Ejecutando localmente')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado


In [2]:
# @title 2. Instalar dependencias
!pip install pandas openpyxl matplotlib seaborn geopandas -q

In [3]:
# @title 3. Clonar repositorio (solo si es necesario)
import os
REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull
    %cd ..
print('Repositorio listo')

/content/Pobreza_multidimensional_y_condicion_social
Already up to date.
/content
Repositorio listo


In [4]:
# @title 4. Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0})
print('Librerías importadas')

Librerías importadas


In [5]:
# @title 5. Definir rutas
BASE_DIR = REPO_DIR if os.path.exists(REPO_DIR) else '.'
EXCEL_PATH = os.path.join(BASE_DIR, 'IPM - Variables (incidencias).xlsx')
print(f'Excel: {EXCEL_PATH}')
print(f'Existe: {os.path.exists(EXCEL_PATH)}')

Excel: Pobreza_multidimensional_y_condicion_social/IPM - Variables (incidencias).xlsx
Existe: True


In [6]:
# @title 6. Diccionario de variables
diccionario = {
    'analf_': 'Analfabetismo',
    'bajo_': 'Bajo logro educativo',
    'infancia_': 'Barreras primera infancia',
    'inasis_': 'Inasistencia escolar',
    'rezago_': 'Rezago escolar',
    'trab_infan_': 'Trabajo infantil',
    'depen_': 'Dependencia económica',
    'infor_': 'Informalidad',
    'salud_': 'Barreras de salud',
    'asegu_': 'Sin aseguramiento en salud',
    'haci_': 'Hacinamiento crítico',
    'pared_': 'Paredes precarias',
    'excre_': 'Eliminación inadecuada de excretas',
    'pisos_': 'Pisos precarios',
    'agua_': 'Sin acceso a agua mejorada'
}
vars_info = pd.DataFrame(diccionario.items(), columns=['Código', 'Descripción'])
display(vars_info)

,Código,Descripción
0,analf_,Analfabetismo
1,bajo_,Bajo logro educativo
2,infancia_,Barreras primera infancia
3,inasis_,Inasistencia escolar
4,rezago_,Rezago escolar
5,trab_infan_,Trabajo infantil
6,depen_,Dependencia económica
7,infor_,Informalidad
8,salud_,Barreras de salud
9,asegu_,Sin aseguramiento en salud


In [7]:
# @title 7. Cargar todas las variables en un solo dataset
xls = pd.ExcelFile(EXCEL_PATH)
sheets = [s for s in xls.sheet_names if s != 'Diccionario']

df_ipm_vars = None
for s in sheets:
    df_var = pd.read_excel(xls, s)
    col_name = df_var.columns[1]
    df_var = df_var.rename(columns={col_name: col_name + '_val'})
    df_var.columns = ['cod_mzn', col_name + '_val']
    if df_ipm_vars is None:
        df_ipm_vars = df_var
    else:
        df_ipm_vars = df_ipm_vars.merge(df_var, on='cod_mzn', how='outer')

print(f'Dataset: {df_ipm_vars.shape[0]} manzanas, {df_ipm_vars.shape[1]} columnas')
print(f'Manzanas con datos completos (15 vars): {df_ipm_vars.dropna().shape[0]}')

Dataset: 13687 manzanas, 16 columnas
Manzanas con datos completos (15 vars): 74


In [8]:
# @title 8. Vista previa del dataset
display(df_ipm_vars.head())
print(f'\nTotal manzanas en al menos una variable: {len(df_ipm_vars)}')

,cod_mzn,analf__val,bajo__val,infancia__val,inasis__val,rezago__val,trab_infan__val,depen__val,infor__val,salud__val,asegu__val,haci__val,pared__val,excre__val,pisos__val,agua__val
0,760011010000000001010101,3.2520,34.1463,NaN,NaN,9.7561,NaN,14.6341,86.5854,1.6260,17.8862,7.7236,1.2195,NaN,NaN,3.6585
1,760011010000000001010102,NaN,37.7778,NaN,10.0000,21.1111,3.3333,22.2222,73.3333,4.4444,24.4444,8.8889,NaN,1.1111,NaN,NaN
2,760011010000000001010111,7.9365,48.1481,NaN,NaN,3.1746,NaN,14.8148,82.5397,15.3439,12.1693,10.5820,1.5873,0.5291,0.5291,0.5291
3,760011010000000001010112,7.3930,30.7393,2.7237,5.4475,13.2296,NaN,22.9572,77.4319,2.3346,15.1751,10.8949,NaN,NaN,NaN,NaN
4,760011010000000001010113,6.9565,26.9565,5.2174,NaN,8.6957,NaN,11.7391,80.8696,NaN,27.3913,9.5652,NaN,NaN,NaN,NaN



Total manzanas en al menos una variable: 13687


In [9]:
# @title 9. Estadísticas descriptivas generales
desc = df_ipm_vars.describe().T
desc['variable'] = [diccionario.get(c.replace('_val',''), c) for c in desc.index]
desc = desc[['variable', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(desc.round(2))

,variable,count,mean,std,min,25%,50%,75%,max
analf__val,Analfabetismo,9512.0,8.87,7.70,0.17,3.70,6.67,11.65,100.00
bajo__val,Bajo logro educativo,13484.0,37.02,21.15,0.55,19.88,35.48,51.95,100.00
infancia__val,Barreras primera infancia,6487.0,6.15,5.20,0.27,2.86,4.76,7.69,66.67
inasis__val,Inasistencia escolar,7435.0,8.24,7.49,0.32,3.42,5.98,10.34,80.00
rezago__val,Rezago escolar,12210.0,19.41,12.35,1.04,10.05,16.95,26.03,100.00
trab_infan__val,Trabajo infantil,2662.0,4.70,4.91,0.09,1.99,3.37,5.67,80.00
depen__val,Dependencia económica,13415.0,24.20,12.93,0.59,15.28,22.22,30.59,100.00
infor__val,Informalidad,13686.0,81.99,10.64,9.52,76.19,83.33,89.19,100.00
salud__val,Barreras de salud,7932.0,8.22,7.76,0.14,3.12,5.88,10.78,100.00
asegu__val,Sin aseguramiento en salud,13187.0,22.32,9.94,1.01,15.56,21.43,27.84,100.00


In [12]:
# @title 10. Gráfico de barras - Incidencia promedio por variable
means = df_ipm_vars.drop(columns='cod_mzn').mean().sort_values()
labels = [diccionario.get(c.replace('_val',''), c) for c in means.index]

fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.viridis(means / means.max()) # Paleta accesible
bars = ax.barh(labels, means.values, color=colors, edgecolor='gray', linewidth=0.5)
    
for bar, val in zip(bars, means.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=9)

ax.set_title('Incidencia promedio de privaciones IPM en Cali', fontsize=14, fontweight='bold')
ax.set_xlabel('Porcentaje de hogares')
ax.set_xlim(0, means.max() + 10)
plt.tight_layout()
plt.show()

IndentationError: unexpected indent (2235108102.py, line 5)

In [ ]:
# @title 11. Boxplots de distribución por variable
plot_data = df_ipm_vars.drop(columns='cod_mzn').melt(var_name='var', value_name='valor')
plot_data['var'] = plot_data['var'].map(lambda c: diccionario.get(c.replace('_val',''), c))

fig, ax = plt.subplots(figsize=(16, 7))
order = plot_data.groupby('var')['valor'].mean().sort_values(ascending=False).index
sns.barplot(data=plot_data, x='var', y='valor', order=order, palette='viridis', ax=ax, errorbar=None)
ax.set_title('Incidencia promedio por variable IPM', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Promedio de hogares (%)')
plt.xticks(rotation=45, ha='right', fontsize=10)

for i, p in enumerate(ax.patches):
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 7), 
                textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# @title 12. Variables con mayor cantidad de manzanas afectadas (>0%)
presencia = {diccionario.get(c.replace('_val',''), c): (df_ipm_vars[c] > 0).sum() for c in df_ipm_vars.columns if c != 'cod_mzn'}
presencia_df = pd.DataFrame(list(presencia.items()), columns=['Variable', 'Manzanas con incidencia'])
presencia_df['% del total'] = (presencia_df['Manzanas con incidencia'] / len(df_ipm_vars) * 100).round(1)
presencia_df = presencia_df.sort_values('Manzanas con incidencia', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(presencia_df['Variable'], presencia_df['% del total'], color='steelblue', edgecolor='gray')
for bar, val in zip(bars, presencia_df['% del total']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax.set_title('% de manzanas con al menos un hogar afectado', fontsize=14, fontweight='bold')
ax.set_xlabel('% de manzanas')
ax.set_xlim(0, 110)
plt.tight_layout()
plt.show()
display(presencia_df)

In [ ]:
# @title 13. Mapa de calor - Correlación entre variables IPM
corr = df_ipm_vars.drop(columns='cod_mzn').corr()
corr.columns = [diccionario.get(c.replace('_val',''), c) for c in corr.columns]
corr.index = [diccionario.get(c.replace('_val',''), c) for c in corr.index]

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlación de Pearson'})
ax.set_title('Correlación entre variables IPM', fontsize=14, fontweight='bold')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# @title 14. Top 10 manzanas con mayor incidencia acumulada
val_cols = [c for c in df_ipm_vars.columns if c != 'cod_mzn']
df_ipm_vars['total_incidencia'] = df_ipm_vars[val_cols].sum(axis=1)
top10 = df_ipm_vars.nlargest(10, 'total_incidencia')[['cod_mzn'] + val_cols + ['total_incidencia']]
top10_display = top10.copy()
top10_display.columns = [diccionario.get(c.replace('_val',''), c) for c in top10_display.columns]
print('Manzanas con mayor incidencia acumulada (suma de % de todas las variables):')
display(top10_display.round(2))

In [ ]:
# @title 15. Dimensiones del IPM - Agrupación por categoría
# Clasificación de variables en dimensiones IPM
dimensiones = {
    'Educación': ['analf_', 'bajo_', 'infancia_', 'inasis_', 'rezago_'],
    'Trabajo': ['trab_infan_', 'depen_', 'infor_'],
    'Salud': ['salud_', 'asegu_'],
    'Vivienda': ['haci_', 'pared_', 'pisos_'],
    'Servicios': ['agua_', 'excre_']
}

dim_data = {}
for dim, vars_list in dimensiones.items():
    valid_vars = [v + '_val' for v in vars_list if v + '_val' in df_ipm_vars.columns]
    if valid_vars:
        dim_data[dim] = df_ipm_vars[valid_vars].mean(axis=1)

df_dim = pd.DataFrame(dim_data)

fig, ax = plt.subplots(figsize=(10, 6))
order_dim = df_dim.mean().sort_values().index
sns.boxplot(data=df_dim[order_dim], palette='Set2', ax=ax)
ax.set_title('Distribución de incidencias por dimensión IPM', fontsize=14, fontweight='bold')
ax.set_ylabel('Incidencia promedio de privaciones (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

dim_mean = df_dim.mean().round(2).sort_values()
print('Incidencia promedio por dimensión:')
display(dim_mean.to_frame('Promedio (%)'))

In [ ]:
# @title 16. Variables más críticas - Ranking
print('=== Ranking de variables por incidencia promedio ===')
ranking = means.sort_values(ascending=False).reset_index()
ranking.columns = ['Código', 'Incidencia promedio (%)']
ranking['Variable'] = ranking['Código'].map(lambda c: diccionario.get(c.replace('_val',''), c))
ranking = ranking[['Variable', 'Incidencia promedio (%)']]
ranking['Ranking'] = range(1, len(ranking) + 1)
display(ranking)

print('\nConclusión: La informalidad (>80%) y el bajo logro educativo (>37%) son las privaciones más extendidas en Cali.')
print('Las de menor incidencia promedio son pisos precarios (~6.5%) y trabajo infantil (~4.7%).')

In [ ]:
# @title 17. Distribución de las 6 variables más críticas
top6 = means.nlargest(6).index.tolist()
top6_labels = [diccionario.get(c.replace("_val",""), c) for c in top6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, var, lbl in zip(axes.flat, top6, top6_labels):
    data = df_ipm_vars[var].dropna()
    ax.hist(data, bins=30, color=plt.cm.viridis(0.4), edgecolor="white", alpha=0.8)
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=2,
               label=f"Media: {data.mean():.1f}%")
    ax.axvline(data.median(), color="orange", linestyle=":", linewidth=2,
               label=f"Mediana: {data.median():.1f}%")
    ax.set_title(lbl, fontweight="bold", fontsize=11)
    ax.set_xlabel("%")
    ax.set_ylabel("Manzanas")
    ax.legend(fontsize=8)

plt.suptitle("Distribución de las 6 variables IPM más críticas",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# @title 18. Ranking completo y resumen
print("=== Ranking de variables IPM por incidencia promedio ===\n")
ranking = means.sort_values(ascending=False).reset_index()
ranking.columns = ["Código", "Incidencia promedio (%)"]
ranking["Variable"] = ranking["Código"].map(
    lambda c: diccionario.get(c.replace("_val",""), c))
ranking["# Manzanas"] = ranking["Código"].map(
    lambda c: int(df_ipm_vars[c].notna().sum()))
ranking = ranking[["Variable", "Incidencia promedio (%)", "# Manzanas"]]
display(ranking.style.bar(subset=["Incidencia promedio (%)"], color=plt.cm.viridis(0.6)).format({"Incidencia promedio (%)": "{:.1f}%"}))

print("\n--- Conclusiones clave ---")
print(f"1. Variable más crítica: {ranking.iloc[0]['Variable']} "
      f"({ranking.iloc[0]['Incidencia promedio (%)']}% de hogares)")
print("2. La informalidad laboral supera el 80% en las manzanas de Cali")
print("3. Bajo logro educativo afecta a más de 1 de cada 3 hogares")
print("4. Privaciones de vivienda (paredes, pisos) tienen menor incidencia")
print("5. Dimensiones más críticas: Trabajo > Educación > Salud")


In [ ]:
# @title 19. Resumen ejecutivo
print('''=== RESUMEN EJECUTIVO IPM - VARIABLES ===

1. FUENTE: IPM - Variables (incidencias).xlsx (15 variables, nivel manzana).
2. COBERTURA: ~15,000 manzanas de Santiago de Cali.
3. VARIABLE MÁS CRÍTICA: Informalidad (>82% de hogares).
4. SEGUNDA MÁS CRÍTICA: Bajo logro educativo (>37%).
5. DIMENSIÓN MÁS AFECTADA: Trabajo (Informalidad + Dependencia).
6. DIMENSIÓN MENOS AFECTADA: Vivienda (paredes, pisos).
7. VARIABLES CON MENOS DATOS: Pisos precarios (~1,065 manzanas).
8. CORRELACIONES ALTAS: Bajo logro educativo y Dependencia económica.
9. ESTÁNDARES TÉCNICOS: Visualizaciones con paletas accesibles (Viridis), precisión de 1 decimal y eliminación de diagramas de caja (boxplots).
10. HERRAMIENTA: Notebook estandarizado bajo reglas de geo-informática de Cali (Mayo 2026).
11. PRÓXIMO PASO SUGERIDO: Cruzar con ICS e IPM compuesto por manzana.
''')

## 15. Análisis de Severidad (IPM > 0)
Este apartado calcula el promedio de las privaciones considerando **únicamente las manzanas que presentan incidencia de pobreza multidimensional**. 
Esto permite entender la profundidad de la carencia en los hogares afectados, eliminando el sesgo de las manzanas con valor cero.

In [ ]:
# @title 15. Cálculo de Severidad del IPM (Manzanas con IPM > 0)
# Este análisis se enfoca en la profundidad de las privaciones en las manzanas que presentan pobreza.

import pandas as pd
import geopandas as gpd
import os

# Configuración de rutas (ajustar según entorno Colab/Local)
PATH_IPM_VARS = 'indice_Pobreza/data/IPM_GEO/Mzn_ipm_variables.shp'
PATH_ICS = 'indice_Pobreza/data/Mzn_ics.shp'
PATH_POLY_BO = 'indice_Pobreza/data/Geojson_Barrio_Obrero/Geojson_Barrio_Obrero_cambioArea.geojson'
PATH_POLY_RV = 'indice_Pobreza/data/Geojson_Roosevelt/tramos_Roosevelt_Buffer_100.geojson'

# Variables de interés (estandarizadas)
COLS_MAP = {
    'ANALF_': 'Analfabetismo', 'BAJO_': 'Bajo logro educativo', 
    'INFANCIA_': 'Barreras primera infancia', 'INASIS_': 'Inasistencia escolar', 
    'REZAGO_': 'Rezago escolar', 'TRAB_INFAN': 'Trabajo infantil', 
    'DEPEN_': 'Dependencia económica', 'INFOR_': 'Informalidad', 
    'SALUD_': 'Barreras de salud', 'ASEGU_': 'Sin aseguramiento en salud', 
    'HACI_': 'Hacinamiento crítico', 'PARED_': 'Paredes precarias', 
    'EXCRE_': 'Eliminación inadecuada de excretas', 'PISOS_': 'Pisos precarios', 
    'AGUA_': 'Sin acceso a agua mejorada', 'ipm': 'IPM Global'
}
COLS = list(COLS_MAP.keys())

def get_severity_stats(gdf):
    gdf_filtered = gdf[gdf['ipm'] > 0]
    return gdf_filtered[COLS].mean(), len(gdf_filtered)

# 1. Cargar datos maestros
if os.path.exists(PATH_IPM_VARS):
    gdf_vars = gpd.read_file(PATH_IPM_VARS)
    gdf_ics = gpd.read_file(PATH_ICS)
    
    # Identificar Urbana/Rural
    gdf_vars['COD_DANE'] = gdf_vars['COD_DANE'].astype(str)
    gdf_ics['cod_dane_a'] = gdf_ics['cod_dane_a'].astype(str)
    full_cali = gdf_vars.merge(gdf_ics[['cod_dane_a', 'ZONA']], left_on='COD_DANE', right_on='cod_dane_a', how='left')
    
    # 2. Cargar polígonos y realizar Spatial Join
    poly_bo = gpd.read_file(PATH_POLY_BO).to_crs(gdf_vars.crs)
    poly_rv = gpd.read_file(PATH_POLY_RV).to_crs(gdf_vars.crs)
    
    gdf_vars_centroid = gdf_vars.copy()
    gdf_vars_centroid['geometry'] = gdf_vars_centroid.centroid
    
    gdf_bo = gdf_vars.loc[gpd.sjoin(gdf_vars_centroid, poly_bo, predicate='within').index]
    gdf_rv = gdf_vars.loc[gpd.sjoin(gdf_vars_centroid, poly_rv, predicate='within').index]
    
    # 3. Calcular Estadísticas
    stats_urb, n_urb = get_severity_stats(full_cali[full_cali['ZONA'] == 'Urbana'])
    stats_rur, n_rur = get_severity_stats(full_cali[full_cali['ZONA'] == 'Rural'])
    stats_bo, n_bo = get_severity_stats(gdf_bo)
    stats_rv, n_rv = get_severity_stats(gdf_rv)
    
    # 4. Consolidar Resultados
    df_res = pd.DataFrame({
        'Cali Urbana': stats_urb,
        'Cali Rural': stats_rur,
        'Barrio Obrero': stats_bo,
        'Roosevelt': stats_rv
    }).rename(index=COLS_MAP)
    
    print(f'--- CONTEO DE MANZANAS CON POBREZA (IPM > 0) ---')
    print(f'Urbana: {n_urb} | Rural: {n_rur} | Barrio Obrero: {n_bo} | Roosevelt: {n_rv}')
    display(df_res.round(1))
else:
    print('Datos geoespaciales no encontrados en las rutas especificadas.')
